In [ ]:
import csv
import os
import random
import re
import subprocess
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif as MIC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Lasso, ElasticNet
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import RFECV

# Seed for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Seed set to:", SEED)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Seed set to: 42
CUDA available: False
Device: cpu
Device name: CPU


# Load the data from Supabase

In [21]:
load_dotenv(Path.cwd().parent / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

TABLE = "lablr"
PAGE_SIZE = 1000  # PostgREST per-request limit

rows = []
start = 0
while True:
    resp = (
        supabase.table(TABLE)
        .select("*")
        .range(start, start + PAGE_SIZE - 1)
        .execute()
    )
    batch = resp.data
    if not batch:
        break
    rows.extend(batch)
    if len(batch) < PAGE_SIZE:
        break
    start += PAGE_SIZE

df_lablr = pd.DataFrame(rows)
print(f"Pulled {len(df_lablr)} rows ({start} pages) from '{TABLE}'")

Pulled 412 rows (0 pages) from 'lablr'


In [22]:
df_lablr.head()

,id,entry_id,title,source_name,assigned_subsector,geography_scope,exec_summary,content,subsector_data,raw,approved,chosen_subsector,is_reclassified,reviewer_id,created_at
0,113,02e851f228f1938f,http://www.tallahassee.com/story/news/world/20...,tallahassee.com,natural_disaster,Garissa County,Armed terrorists stormed a university in north...,"NAIROBI, Kenya — Armed terrorists stormed a un...","{'beds_offline': 0, 'disaster_name': 'Garissa ...","{'id': '02e851f228f1938f', 'title': 'http://ww...",False,NaN,False,Edgar,2026-06-29T00:53:57.236902+00:00
1,119,057f6d5019c6ccc2,FDA will destroy banned imported drugs,modernhealthcare.com,drug_shortage,US,FDA will destroy banned imported drugs,Gift Article\n\nShare\n\nBy\n\nSteven Ross Joh...,"{'drug_name': None, 'dosage_form': None, 'gene...","{'id': '057f6d5019c6ccc2', 'title': 'FDA will ...",False,NaN,False,Edgar,2026-06-29T01:06:08.672704+00:00
2,211,25f2020ce1c3b46a,Colorado rejects medical marijuana for PTSD tr...,rawstory.com,cyber_attack,US,,2026 Midterms\n\nUS NEWS\n\nInvestigations\n\n...,"{'attack_type': None, 'ransom_paid': None, 'th...","{'id': '25f2020ce1c3b46a', 'title': 'Colorado ...",False,NaN,False,Edgar,2026-06-29T19:25:41.319934+00:00
3,213,1bbbc7a53775abe4,"Med students, volunteer pros staff Mollie Whea...",tribstar.com,natural_disaster,NaN,,You are the owner of this article.\n\nEdit Art...,"{'beds_offline': None, 'disaster_name': 'Molli...","{'id': '1bbbc7a53775abe4', 'title': 'Med stude...",False,NaN,False,Dolan,2026-06-29T19:26:07.558386+00:00
4,219,1de21fd03b915dc0,Hackers accessed patients' credit card informa...,missoulian.com,cyber_attack,Montana,Hackers accessed patients' credit card informa...,"Missoulian\n\n, P.O. Box 8029 Missoula, MT 598...","{'attack_type': None, 'ransom_paid': None, 'th...","{'id': '1de21fd03b915dc0', 'title': 'Hackers a...",False,NaN,False,Dolan,2026-06-29T19:26:27.544686+00:00


### Consts and helper functions

In [ ]:
CLASS_NAMES = ["noise", "drug_shortage", "medical_device_shortage", "cyber_attack", "natural_disaster", "other"]

LABEL_MAP = {
    "noise": 0,
    "drug_shortage": 1,
    "medical_device_shortage": 2,
    "cyber_attack": 3,
    "natural_disaster": 4,
    "other": 5
}

IDX_TO_LABEL = {
    0: "noise",
    1: "drug_shortage",
    2: "medical_device_shortage",
    3: "cyber_attack",
    4: "natural_disaster",
    5: "other"
}

DROP_COLS = [
    "entry_id", "assigned_subsector", "raw", "chosen_subsector",
    "is_reclassified", "reviewer_id", "created_at", "approved"
]

X_COLS= ["title", "source_name", "geography_scope",	"exec_summary",	"content", "subsector_data"]

def _make_pipe(clf) -> Pipeline:
    """
    Used to prep every model the same way before training

    Args:
        clf: the different classifiers used
    Returns:
        Pipeline object with the same specs
    """
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")), # swap Nan <-> median
        ("scale", StandardScaler()), # req for LR/SVC/MLP, RF doesn't need it but still gets it
        ("clf", clf),
    ])


def classic_models():
    """
    Returns:
        a fresh dict of NEW, untrained pipeline instances every call
    """
    return {
        "RandomForest": _make_pipe(RandomForestClassifier(random_state=SEED)),
        "LogisticRegression": _make_pipe(LogisticRegression(max_iter=1000, random_state=SEED)),
        "SVC": _make_pipe(SVC(random_state=SEED)),
        "MLP": _make_pipe(MLPClassifier(max_iter=1000, random_state=SEED)),
    }
# TODO: add more models here + bert 

def _derive_target(row):
    """
    Make a new label and class given a supabase row

    Args:
        row (straight from supabase)

    Returns:
        'label' and 'map' series with the correct classification
        If 'chosen_subsector' is not in LABEL_MAP returns nan pair
    """
    if not row["approved"]:
        return pd.Series({"label": 0, "class": "noise"})
    sub = row["chosen_subsector"]
    if sub in LABEL_MAP:
        return pd.Series({"label": LABEL_MAP[sub], "class": sub})
    return pd.Series({"label": np.nan, "class": np.nan}) 

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


df = df_lablr.copy()
df[["label", "class"]] = df.apply(_derive_target, axis=1) # axis 1 needed for cols

# drop unmapped edge-case rows, then lock label to int
df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(int) # make sure label is int
df = df.drop(columns=DROP_COLS)

X = df[X_COLS]
y = df["label"]

df["class"].value_counts()

class
noise                      201
cyber_attack               153
other                       21
natural_disaster            20
drug_shortage               12
medical_device_shortage      5
Name: count, dtype: int64

In [31]:
df.head()

,id,title,source_name,geography_scope,exec_summary,content,subsector_data,label,class
0,113,http://www.tallahassee.com/story/news/world/20...,tallahassee.com,Garissa County,Armed terrorists stormed a university in north...,"NAIROBI, Kenya — Armed terrorists stormed a un...","{'beds_offline': 0, 'disaster_name': 'Garissa ...",0,noise
1,119,FDA will destroy banned imported drugs,modernhealthcare.com,US,FDA will destroy banned imported drugs,Gift Article\n\nShare\n\nBy\n\nSteven Ross Joh...,"{'drug_name': None, 'dosage_form': None, 'gene...",0,noise
2,211,Colorado rejects medical marijuana for PTSD tr...,rawstory.com,US,,2026 Midterms\n\nUS NEWS\n\nInvestigations\n\n...,"{'attack_type': None, 'ransom_paid': None, 'th...",0,noise
3,213,"Med students, volunteer pros staff Mollie Whea...",tribstar.com,NaN,,You are the owner of this article.\n\nEdit Art...,"{'beds_offline': None, 'disaster_name': 'Molli...",0,noise
4,219,Hackers accessed patients' credit card informa...,missoulian.com,Montana,Hackers accessed patients' credit card informa...,"Missoulian\n\n, P.O. Box 8029 Missoula, MT 598...","{'attack_type': None, 'ransom_paid': None, 'th...",0,noise


In [37]:
def run_experiment(X, y, models = classic_models()):
  """

  """
  results = {} # name = (mean_acc, std_acc)

  for name, pipe in models.items():
      print("="*75)
      print(name)

      # get a np array of scores
      scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
      results[name] = (scores.mean(), scores.std()) # save 1 score mean with std

      print(f"Accuracy per fold: {np.round(scores, 4)}")
      print(f"Mean accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

      #collects each window's out-of-fold prediction
      y_pred = cross_val_predict(pipe, X, y, cv=cv) # each window's out-of-fold prediction
      print("\nClassification report (out-of-fold):")
      print(classification_report(y, y_pred, target_names=CLASS_NAMES))

      # confusion matrix
      cm = confusion_matrix(y, y_pred, labels=range(len(CLASS_NAMES)))
      plt.figure(figsize=(5, 4))
      sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
      plt.title(f"{name} -- out-of-fold confusion matrix")
      plt.xlabel("Predicted")
      plt.ylabel("True")
      plt.show()
      print("="*75)


  print("Summary -- algorithms ranked by mean 5-fold accuracy")
  print("-"*60)
  ranking = sorted(results.items(), key=lambda kv: kv[1][0], reverse=True)
  for rank, (name, (mean_acc, std_acc)) in enumerate(ranking, start=1):
    print(f"{rank}. {name:<20} {mean_acc:.4f} ± {std_acc:.4f}")
    

In [ ]:
run_experiment(X,y)